# Extraction de données multimodales — Google Fact Check Tools API

Ce notebook interroge l'API **Claims Search** de Google Fact Check Tools pour collecter des affirmations vérifiées (fact-checks), puis enrichit chaque résultat avec le texte complet de l'article de vérification, afin de constituer un jeu de données **texte + label** (`fake`/`true`) exploitable pour l'entraînement d'un modèle.

## Sommaire
1. Configuration
2. Client API — recherche de claims
3. Collecte paginée
4. Mise en dataframe
5. Exécution de la collecte, filtrage par date, normalisation du label
6. Enrichissement — texte de l'article source
7. Sauvegarde du dataset consolidé
8. Exploration rapide

## Prérequis
- Une clé API Google Cloud avec la **Fact Check Tools API** activée.
- La clé placée dans un fichier `.env` à la racine du projet : `GOOGLE_FACTCHECK_API_KEY=...`

## 1. Configuration

In [37]:
import os
import re
import time
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.auto import tqdm

In [ ]:
load_dotenv()

API_KEY = os.environ.get("GOOGLE_FACTCHECK_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "Missing GOOGLE_FACTCHECK_API_KEY. Set it in a .env file at the project root."
    )

CLAIMS_SEARCH_URL = "https://factchecktools.googleapis.com/v1alpha1/claims:search"

# Output locations
PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "google_factcheck"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Be polite with external services
REQUEST_DELAY_SECONDS = 0.1
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; Projet12-FactCheckCollector/1.0)"}

# Only keep fact-checks published within this many days: older reviews are more likely
# to point to dead/invalid links
MAX_AGE_DAYS = 180

## 2. Client API — recherche de claims

Wrapper autour de l'endpoint `claims:search`, avec gestion des erreurs HTTP et du throttling.

In [39]:
def search_claims(
    query: str | None = None,
    language_code: str | None = None,
    page_size: int = 1,
    page_token: str | None = None,
    max_age_days: int | None = None,
    review_publisher_site_filter: str | None = None,
) -> dict:
    """Call the Google Fact Check Tools claims:search endpoint once.

    Either query or review_publisher_site_filter must be provided (the API requires one).
    language_code left unset (None) searches all languages.
    """
    if not query and not review_publisher_site_filter:
        raise ValueError("Provide at least one of query or review_publisher_site_filter")

    params = {
        "key": API_KEY,
        "pageSize": page_size,
    }
    if query:
        params["query"] = query
    if language_code:
        params["languageCode"] = language_code
    if page_token:
        params["pageToken"] = page_token
    if max_age_days:
        params["maxAgeDays"] = max_age_days
    if review_publisher_site_filter:
        params["reviewPublisherSiteFilter"] = review_publisher_site_filter

    response = requests.get(CLAIMS_SEARCH_URL, params=params, timeout=30)
    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        # Surface Google's actual error body (raise_for_status only gives the status code)
        raise requests.HTTPError(f"{error}\nResponse body: {response.text}") from error
    return response.json()

## 3. Collecte paginée

On suit les `nextPageToken` jusqu'à épuisement ou jusqu'à une limite de résultats, en filtrant sur `maxAgeDays` pour ne récupérer que des fact-checks récents (les liens plus anciens sont souvent morts).

In [40]:
def collect_claims(
    query: str | None = None,
    language_code: str | None = None,
    max_results: int = 300,
    page_size: int = 20,
    max_age_days: int | None = MAX_AGE_DAYS,
    review_publisher_site_filter: str | None = None,
) -> list[dict]:
    """Collect raw claim records for a single query and/or publisher site, following pagination."""
    all_claims: list[dict] = []
    page_token = None

    with tqdm(total=max_results, desc=review_publisher_site_filter or query) as progress_bar:
        while True:
            payload = search_claims(
                query=query,
                language_code=language_code,
                page_size=page_size,
                page_token=page_token,
                max_age_days=max_age_days,
                review_publisher_site_filter=review_publisher_site_filter,
            )
            claims = payload.get("claims", [])
            all_claims.extend(claims)
            progress_bar.update(len(claims))

            page_token = payload.get("nextPageToken")
            time.sleep(REQUEST_DELAY_SECONDS)

            if not page_token or len(all_claims) >= max_results or not claims:
                break

    return all_claims

## 4. Mise en dataframe

Chaque claim peut contenir plusieurs `claimReview` (une par média de vérification) : on met les résultats à plat, une ligne par couple claim/review.

In [41]:
def claims_to_dataframe(raw_claims: list[dict]) -> pd.DataFrame:
    """Flatten raw claims:search results into one row per claim review."""
    rows = []
    for claim in raw_claims:
        for review in claim.get("claimReview", []):
            publisher = review.get("publisher", {})
            rows.append(
                {
                    "claim_text": claim.get("text"),
                    "claimant": claim.get("claimant"),
                    "claim_date": claim.get("claimDate"),
                    "review_publisher_name": publisher.get("name"),
                    "review_publisher_site": publisher.get("site"),
                    "review_url": review.get("url"),
                    "review_title": review.get("title"),
                    "review_date": review.get("reviewDate"),
                    "textual_rating": review.get("textualRating"),
                    "language_code": review.get("languageCode"),
                }
            )
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["review_url"]).reset_index(drop=True)
    return df

## 5. Exécution de la collecte

Filtrer par mot-clé (`query`) restreint trop le volume — on interroge à la place directement une liste de fact-checkers connus via `reviewPublisherSiteFilter`, sans aucun filtre thématique, puis on fusionne et déduplique les résultats.

In [42]:
FACT_CHECK_SITES = [
    "politifact.com",
    "snopes.com",
    "factcheck.org",
    "fullfact.org",
    "leadstories.com",
    "factcheck.afp.com",
    "reuters.com",
    "apnews.com",
    "africacheck.org",
    "boomlive.in",
    "altnews.in",
    "correctiv.org",
    "maldita.es",
    "newtral.es",
    "rappler.com",
    "verafiles.org",
]
PER_SITE_MAX_RESULTS = 100

raw_claims = []
for site in FACT_CHECK_SITES:
    raw_claims.extend(
        collect_claims(
            review_publisher_site_filter=site,
            max_results=PER_SITE_MAX_RESULTS,
            max_age_days=MAX_AGE_DAYS,
        )
    )
print(f"Collected {len(raw_claims)} raw claims across {len(FACT_CHECK_SITES)} publishers")

df = claims_to_dataframe(raw_claims)
print(f"Flattened into {len(df)} unique claim reviews")
df.head()

politifact.com:   0%|          | 0/100 [00:00<?, ?it/s]

snopes.com:   0%|          | 0/100 [00:00<?, ?it/s]

factcheck.org:   0%|          | 0/100 [00:00<?, ?it/s]

fullfact.org:   0%|          | 0/100 [00:00<?, ?it/s]

leadstories.com:   0%|          | 0/100 [00:00<?, ?it/s]

...

newtral.es:   0%|          | 0/100 [00:00<?, ?it/s]

rappler.com:   0%|          | 0/100 [00:00<?, ?it/s]

verafiles.org:   0%|          | 0/100 [00:00<?, ?it/s]

Collected 1079 raw claims across 16 publishers
Flattened into 1061 unique claim reviews


,claim_text,claimant,claim_date,review_publisher_name,review_publisher_site,review_url,review_title,review_date,textual_rating,language_code
0,"The AI industry needs ""more energy than the en...",Donald Trump,2026-07-08T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jul/14/...,"Data centers need a lot of energy, but Preside...",2026-07-14T00:00:00Z,Mostly False,en
1,"""You know why I'm profiting? Because the stock...",Donald Trump,2026-07-01T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jul/07/...,Trump says stock market drove his 2025 income ...,2026-07-07T00:00:00Z,Mostly False,en
2,Says benefits of the 21st Century ROAD to Hous...,Tommy Tuberville,2026-06-24T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jun/26/...,Is the bipartisan housing bill for “a lot of p...,2026-06-26T00:00:00Z,False,en
3,"Sen. Susan Collins “sided with Trump, giving b...",Graham Platner,2026-06-17T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jun/25/...,Platner says Collins sided with Trump to cut M...,2026-06-25T00:00:00Z,False,en
4,"In the past 13 months, “The typical 401(k) … i...",Donald Trump,2026-06-23T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jun/25/...,Donald Trump exaggerates 401(k) gains during h...,2026-06-25T00:00:00Z,Mostly False,en


## 5b. Filtrage par date

`maxAgeDays` filtre déjà côté API, mais `reviewDate` n'est pas toujours renseigné ou fiable : on applique un second filtrage explicite ici, et on écarte les lignes sans date exploitable.

In [43]:
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce", utc=True)

cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=MAX_AGE_DAYS)
before = len(df)
df = df[df["review_date"].notna() & (df["review_date"] >= cutoff)].reset_index(drop=True)

print(f"Kept {len(df)}/{before} rows published after {cutoff.date()}")
df.head()

Kept 1049/1061 rows published after 2026-03-14


,claim_text,claimant,claim_date,review_publisher_name,review_publisher_site,review_url,review_title,review_date,textual_rating,language_code
0,"The AI industry needs ""more energy than the en...",Donald Trump,2026-07-08T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jul/14/...,"Data centers need a lot of energy, but Preside...",2026-07-14 00:00:00+00:00,Mostly False,en
1,"""You know why I'm profiting? Because the stock...",Donald Trump,2026-07-01T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jul/07/...,Trump says stock market drove his 2025 income ...,2026-07-07 00:00:00+00:00,Mostly False,en
2,Says benefits of the 21st Century ROAD to Hous...,Tommy Tuberville,2026-06-24T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jun/26/...,Is the bipartisan housing bill for “a lot of p...,2026-06-26 00:00:00+00:00,False,en
3,"Sen. Susan Collins “sided with Trump, giving b...",Graham Platner,2026-06-17T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jun/25/...,Platner says Collins sided with Trump to cut M...,2026-06-25 00:00:00+00:00,False,en
4,"In the past 13 months, “The typical 401(k) … i...",Donald Trump,2026-06-23T00:00:00Z,PolitiFact,politifact.com,https://politifact.com/factchecks/2026/jun/25/...,Donald Trump exaggerates 401(k) gains during h...,2026-06-25 00:00:00+00:00,Mostly False,en


## 5c. Normalisation du label (fake / true)

`textual_rating` est un texte libre choisi par chaque organisme de fact-checking ("Pants on Fire", "Mostly True", "Faux", "Trompeur", ...). Pour entraîner un modèle binaire, on le normalise en un label clair `{"fake", "true"}` à partir de mots-clés, et on **rejette** les évaluations ambiguës (mi-vrai, satire, non vérifiable, ...) plutôt que de les forcer dans une catégorie.

Le déséquilibre observé (très peu de `"true"`) reflète le biais naturel des fact-checkers, qui vérifient surtout des affirmations fausses. On y répond en élargissant la collecte (`max_results` plus grand ci-dessus) plutôt qu'en filtrant davantage ; un rééquilibrage optionnel (sous-échantillonnage de la classe majoritaire) est proposé en fin de section."

In [44]:
import re

# Ratings that are inherently ambiguous for a binary fake/true task: excluded rather than guessed
AMBIGUOUS_RATING_PATTERNS = [
    "half true", "half false", "mixture", "mixed", "unproven", "unverified",
    "opinion", "satire", "outdated", "missing context", "out of context",
    "needs context", "unclear", "research in progress", "altered context",
    "mi-vrai", "mi-faux", "à vérifier", "sans fondement suffisant",
]
FAKE_RATING_PATTERNS = [
    "false", "fake", "pants on fire", "misleading", "incorrect", "not true",
    "no evidence", "no such report", "distort", "manipulated", "hoax", "scam",
    "alter", "miscaption",
    "faux", "trompeur", "mensonge", "inexact", "erroné", "erronee", "falso",
]
TRUE_RATING_PATTERNS = [
    "true", "correct", "accurate", "vrai", "exact", "verified", "confirmed",
]


def normalize_rating(textual_rating: str | float) -> str | None:
    """Map a free-text textual_rating to a clean binary label, or None if ambiguous/unrecognized."""
    if pd.isna(textual_rating):
        return None
    rating = textual_rating.strip().lower()

    if any(re.search(pattern, rating) for pattern in AMBIGUOUS_RATING_PATTERNS):
        return None
    if any(re.search(pattern, rating) for pattern in FAKE_RATING_PATTERNS):
        return "fake"
    if any(re.search(pattern, rating) for pattern in TRUE_RATING_PATTERNS):
        return "true"
    return None

In [45]:
df["label"] = df["textual_rating"].apply(normalize_rating)

before = len(df)
unmapped_ratings = df.loc[df["label"].isna(), "textual_rating"].value_counts()
df = df[df["label"].notna()].reset_index(drop=True)

print(f"Kept {len(df)}/{before} rows with a clear fake/true label")
print("\nClass balance:")
print(df["label"].value_counts())
print("\nTop unmapped ratings (review AMBIGUOUS/FAKE/TRUE_RATING_PATTERNS if this list looks wrong):")
print(unmapped_ratings.head(15))

Kept 772/1049 rows with a clear fake/true label

Class balance:
label
fake    723
true     49
Name: count, dtype: int64

Top unmapped ratings (review AMBIGUOUS/FAKE/TRUE_RATING_PATTERNS if this list looks wrong):
textual_rating
Engañoso                30
Contexto                27
Alerta                  26
Mixture                 18
Originated as Satire    10
Labeled Satire           8
Made With AI             7
Half True                6
Satire Label             6
AI-generated             6
Satire                   5
Unsupported              4
AI-Generated             4
Missing Context          4
Prebunk                  3
Name: count, dtype: int64


## 6. Enrichissement — texte de l'article source

On récupère le texte intégral de la page de vérification (`review_url`) pour disposer du contenu complet de l'article, pas seulement du titre et de l'évaluation.

In [46]:
def fetch_article_text(page_url: str, min_length: int = 200) -> str | None:
    """Fetch a review article page and extract its main body text.

    Returns None on a fetch failure or if the extracted text looks too short
    to be real article content (e.g. a paywall or JS-only page).
    """
    try:
        response = requests.get(page_url, headers=HTTP_HEADERS, timeout=15)
        response.raise_for_status()
    except requests.RequestException:
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    article = soup.find("article") or soup.find("main") or soup.find(attrs={"role": "main"})
    paragraphs = (article or soup).find_all("p")

    text = "\n".join(p.get_text(" ", strip=True) for p in paragraphs)
    text = re.sub(r"\n{2,}", "\n", text).strip()

    return text if len(text) >= min_length else None

In [48]:
article_texts = []
for review_url in tqdm(df["review_url"], desc="Fetching article text"):
    article_texts.append(fetch_article_text(review_url))
    time.sleep(REQUEST_DELAY_SECONDS)

df["article_text"] = article_texts
df["article_text"].notna().mean()

Fetching article text:   0%|          | 0/772 [00:00<?, ?it/s]

np.float64(0.5867875647668394)

## 7. Sauvegarde du dataset consolidé

In [49]:
output_csv = DATA_DIR / "google_factcheck_multimodal.csv"
df.to_csv(output_csv, index=False)
print(f"Saved {len(df)} rows to {output_csv}")

Saved 772 rows to c:\Formation\Projet_12\data\google_factcheck\google_factcheck_multimodal.csv


## 8. Exploration rapide

In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 772 entries, 0 to 771
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype              
---  ------                 --------------  -----              
 0   claim_text             772 non-null    str                
 1   claimant               528 non-null    str                
 2   claim_date             529 non-null    str                
 3   review_publisher_name  772 non-null    str                
 4   review_publisher_site  772 non-null    str                
 5   review_url             772 non-null    str                
 6   review_title           772 non-null    str                
 7   review_date            772 non-null    datetime64[us, UTC]
 8   textual_rating         772 non-null    str                
 9   language_code          772 non-null    str                
 10  label                  772 non-null    str                
 11  article_text           453 non-null    str                
dtypes: da

In [51]:
df["label"].value_counts()

label
fake    723
true     49
Name: count, dtype: int64

In [52]:
df[df["article_text"].notna()][
    ["claim_text", "label", "textual_rating", "review_publisher_name", "article_text"]
].head(10)

,claim_text,label,textual_rating,review_publisher_name,article_text
0,"The AI industry needs ""more energy than the en...",fake,Mostly False,PolitiFact,The AI industry needs “more energy than the en...
1,"""You know why I'm profiting? Because the stock...",fake,Mostly False,PolitiFact,“You know why I’m profiting? Because the stock...
2,Says benefits of the 21st Century ROAD to Hous...,fake,False,PolitiFact,Says benefits of the 21st Century ROAD to Hous...
3,"Sen. Susan Collins “sided with Trump, giving b...",fake,False,PolitiFact,"Sen. Susan Collins “sided with Trump, giving b..."
4,"In the past 13 months, “The typical 401(k) … i...",fake,Mostly False,PolitiFact,"In the past 13 months, “The typical 401(k) … i..."
5,"""The number of New Yorkers who consume cannabi...",fake,Mostly False,PolitiFact,“The number of New Yorkers who consume cannabi...
6,Says his cognitive test results showed he has ...,fake,False,PolitiFact,Says his cognitive test results showed he has ...
7,“President Trump’s tariffs are costing New Yor...,fake,Mostly False,PolitiFact,“President Trump’s tariffs are costing New Yor...
8,"""There's been numerous other occasions over th...",fake,False,PolitiFact,“There’s been numerous other occasions over th...
9,Proposed data centers in New York state would ...,fake,Mostly False,PolitiFact,Proposed data centers in New York state would ...
